# Bilancia and co

In [35]:
import pyvisa
import numpy as np
import time
import serial
from serial.tools import list_ports
import matplotlib.pyplot as plt
import cv2

In [36]:
%matplotlib widget
plt.close('all')

In [ ]:
class Bilancia: 

    comandi = {
        "identificazione" : (chr(255) + chr(254) + chr(1) + chr(1) + chr(0) + chr(254)).encode('latin-1'),
        "iniziazione misure" : (chr(255) + chr(254) + chr(1) + chr(5) + chr(2) + chr(248) + chr(1) + chr(255)).encode('latin-1'),
        "stop" : (chr(255)+chr(254)+chr(1)+chr(5)+chr(2)+chr(0)+chr(0)+chr(248)).encode('latin-1')
    }

    def __init__(self, porta, baudrate=9600, timeout=2):
        self.porta = porta
        self.baudrate = baudrate
        self.timeout = timeout
        self.libra = serial.Serial(self.porta, self.baudrate, timeout=self.timeout)
    
    def reset_input_buffers(self):
        self.libra.reset_input_buffer()
    
    def reset_output_buffers(self):
        self.libra.reset_output_buffer()
    
    def send_command(self, comando):
        if comando in self.comandi:
            self.libra.write(self.comandi[comando])
        else:
            raise ValueError(f"Comando '{comando}' non riconosciuto.")

    def identify(self):
        self.send_command("identificazione")
        risposta = self.libra.read(60)
        return risposta
    
    def start_measurements(self):
        self.send_command("iniziazione misure")
        self.libra.read(8) #leggo corretta ricezione del messaggio di prendere le misure
        print("Misure microbilancia in corso")
    
    def read_measurement(self, libra_data, startime): #andrebbe messo in un ciclo per leggere più misure, ma per ora lo lascio così
        
        self.duration_l = round( time.time() - startime, 5)
        self.libra.read(5) #leggo la testa della singola misura
        rate_string = self.libra.read(5) #something like b'...'
        spessore_string = self.libra.read(6)    
        self.libra.read(32)# leggo la coda della singola misura

        rate = rate_string.decode('latin-1')
        spessore=spessore_string.decode('latin-1')
        libra_data.append((self.duration_l, float(rate),float(spessore))) #libra data è un array di tuple (tempo di acquisizione, rate, spessore)
        
        return float(rate), float(spessore)
    
    def stop_measurements(self, fname, X):
        
        print("Fine delle misure")
        
        np.savetxt(fname, X,fmt='%s', delimiter = '\t', header = "Tempo\t Rate\t Spessore")
        self.reset_input_buffers()
        self.send_command("stop")
        
        self.reset_input_buffers()
        self.reset_output_buffers()

        self.libra.close()

    def __delete__(self, instance):
        self.stop_measurements()    

class Multimetro:

    comandi = {
        "version": "SYST:VERS?",
        "identificazione": '*IDN?',
        "reset": '*RST',
        "imposta in ohm": 'OHMS',
        "autoscale": "AUTO",
        "leggi valore1": 'VAL1?',
    }   

    def __init__(self, resource_manager, resource_name):
        self.resource_manager = resource_manager
        self.multimeter = self.resource_manager.open_resource(resource_name)
        self.multimeter.clear()  # Pulisce i buffer di comunicazione
        self.multimeter.timeout = 10000  
        
        #self.multimeter.baud_rate = 9600
        
        #self.multimeter.read_termination = '\n'
        #self.multimeter.write_termination = '\n'
    
    def query_version(self):
        return self.multimeter.query(self.comandi["version"])
    
    def identify(self):
        return self.multimeter.query(self.comandi["identificazione"])

    def initialize(self):
        self.multimeter.write(self.comandi["reset"])
        time.sleep(1)
        
        self.multimeter.write(self.comandi["autoscale"])
        self.multimeter.write(self.comandi["imposta in ohm"])
    
    def read_value(self):
        value = self.multimeter.query(self.comandi["leggi valore1"])
        return float(value)

    def close(self):
        self.multimeter.close()


class Camera:
    def __init__(self, camera_index=0):
        self.cap = cv2.VideoCapture(camera_index)

        self.im0 = None  # Immagine di riferimento

        if not self.cap.isOpened():
            raise RuntimeError("Impossibile aprire la webcam.")
    
    def set_camera_params(self, exposure=-5, wb_temp=3900):
        self.cap.set(cv2.CAP_PROP_AUTO_EXPOSURE, 0.25)
        self.cap.set(cv2.CAP_PROP_EXPOSURE, exposure)
        self.cap.set(cv2.CAP_PROP_AUTO_WB, 0)
        self.cap.set(cv2.CAP_PROP_WB_TEMPERATURE, wb_temp)
    
    def acquire_image(self):
        ret, frame = self.cap.read()
        if not ret:
            raise RuntimeError("Impossibile acquisire un frame dalla webcam.")
        
        return frame
    
    def build_roi_masks(self, im0, center_x, center_y, radius):
        h, w = im0.shape[:2]
        Y, X = np.ogrid[:h, :w]
        dist_from_center = np.sqrt((X - center_x)**2 + (Y - center_y)**2)
        
        masks = {
            'total': dist_from_center <= radius,
            'mid': dist_from_center <= (radius / 2),
            'in': dist_from_center <= (radius / 4),
            'q1': (dist_from_center <= radius) & (Y < center_y) & (X > center_x),
            'q2': (dist_from_center <= radius) & (Y < center_y) & (X < center_x),
            'q3': (dist_from_center <= radius) & (Y > center_y) & (X < center_x),
            'q4': (dist_from_center <= radius) & (Y > center_y) & (X > center_x)
        }
        
        return masks
    
    def acquire_reference_image(self, avgs=16):
        frames = []
        for _ in range(avgs):
            frame = self.acquire_image()
            frames.append(frame.astype(np.float32))
            time.sleep(0.1) # Breve pausa per dare tempo al sensore
            
        if not frames:
            raise RuntimeError("Acquisizione immagine di riferimento fallita.")
            
        self.im0 = np.mean(frames, axis=0).astype(np.uint8)
        self.im0 = cv2.cvtColor(self.im0, cv2.COLOR_BGR2GRAY)
        
        return self.im0.astype(np.uint8)

    def process_frame(self, im0, masks):

        if im0 is None or not masks:
            raise ValueError("Immagine di riferimento e maschere ROI devono essere inizializzate prima di processare.")

        frame = self.acquire_image()
        im1_float = frame.astype(np.float32)
        diff = im1_float - im0
        heatmap = np.linalg.norm(diff, axis=-1)

        ii = np.mean(255 - diff[masks['total']])
        ii_in = np.mean(255 - diff[masks['in']])
        ii_mid = np.mean(255 - diff[masks['mid']])
        
        ii1 = np.mean(255 - diff[masks['q1']])
        ii2 = np.mean(255 - diff[masks['q2']])
        ii3 = np.mean(255 - diff[masks['q3']])
        ii4 = np.mean(255 - diff[masks['q4']])
        
        return frame, heatmap, ii, ii_in, ii_mid, ii1, ii2, ii3, ii4

    def release(self):
        self.cap.release()
    
    def __delete__(self, instance):
        self.release()



In [ ]:

class SCPIInstrument:
    """
        Representation of a generic SCPI instrument communicating via PySerial.
    """

    commands = {
        "identify": "*IDN?",
        "reset": "*RST",
        "clear": "*CLS"
    }
    
    def __init__(self, port, baudrate=9600, timeout=2, terminator='\r\n'):

        self.port = port
        self.baudrate = baudrate
        self.timeout = timeout
        self.terminator = terminator
        
        # Init the serial connection
        self.serial = serial.Serial(self.port, self.baudrate, timeout=self.timeout)
        self.reset_buffers()

    def reset_buffers(self):
        """Reset internal buffers"""
        self.serial.reset_input_buffer()
        self.serial.reset_output_buffer()

    def send_command(self, command):
        """Sends a raw SCPI command (automatically appends the terminator)."""
        complete_command = f"{command}{self.terminator}"
        self.serial.write(complete_command.encode('ascii'))

    def query(self, command, delay=0.05):
        """Sends a query and reads the instrument's response."""
        self.send_command(command)
        time.sleep(delay)
        response = self.serial.readline() 
        # con readline(), dopo la lettura di una misura, il buffer del pc la elimina (non usiamo il buffer del keithley grazie alla funzione READ?)
        
        # decode and clean up the response
        return response.decode('ascii', errors='ignore').strip()

    def identify(self):
        """Sends a universal SCPI command to identify the instrument."""
        return self.query(command=self.commands["identify"])

    def reset(self):
        """Sends a universal SCPI command to reset the instrument to factory defaults."""
        self.send_command(self.commands["reset"])
        self.send_command(self.commands["clear"])

    def close(self):
        """Sends a command to close the communication cleanly."""
        if self.serial.is_open:
            self.serial.close()

    def __delete__(self, instance):
        """Ensure the serial connection is closed when the object is deleted."""
        self.close()

class ElettrometroKeithley(SCPIInstrument):
    """Implementation for the Keithley 6517A."""

    # specific SCPI commands for the Keithley 6517A
    commands = {
        "zero_check_on": "SYST:ZCH ON", #relè attivo
        "zero_check_off": "SYST:ZCH OFF", #relè disattivo
        "configure_current": "CONF:CURR:DC",
        "configure_resistance": "CONF:RES",          
        "format_elements": "FORM:ELEM READ,TIME",    # chiede allo strumento: Lettura e Tempo
        "reset_time": "SYST:TIME:RES"                # azzera il timer interno 


    }
    
    def __init__(self, port, baudrate=9600, timeout=2):
        # create a serial connection with the correct terminator for the Keithley (typically \r\n)
        super().__init__(port, baudrate, timeout, terminator='\r\n')

        # combine the base class commands with the Keithley-specific commands
        self.commands = {**SCPIInstrument.commands, **self.__class__.commands}
        
    def set_zero_check(self, state: bool):
        """Attiva (True) o disattiva (False) lo Zero Check."""
        cmd = self.commands["zero_check_on"] if state else self.commands["zero_check_off"]
        self.send_command(cmd)
        
    def get_fresh_reading(self):
        """Invia READ? e restituisce la stringa grezza senza toccare i relè."""
        return self.query("READ?")
    
    def parse_resistance_reading(self, raw_value):
        """Funzione di utility per pulire i dati (da chiamare nel ciclo)."""
        try:
            parts = raw_value.split(',')
            res_val = float(parts[0].replace('OHM', '').replace('A', '').replace('V', ''))
            time_val = float(parts[1].replace('secs', ''))
            return res_val, time_val
        except (ValueError, IndexError):
            return None, None

    def init_current_reading(self):
        """Set up the electrometer in a safe way to read currents."""
        self.reset()
        time.sleep(0.5)
        self.send_command(self.commands["zero_check_on"])
        self.send_command(self.commands["configure_current"])
"""
    def read_current(self):
        """Reads a single current measurement."""
        self.send_command(self.commands["zero_check_off"])
        time.sleep(0.1) 
        
        raw_value = self.query("READ?")
        
        self.send_command(self.commands["zero_check_on"])
        
        try:
            # Il Keithley restituisce stringhe tipo "+1.23456E-09A, +000..."
            clean_value = float(raw_value.split(',')[0].replace('A', ''))
            return clean_value
        except (ValueError, IndexError):
            return raw_value  # Return the raw value for debugging if parsing fails
    """
    
        
    def init_resistance_reading(self):
            """Prepara l'elettrometro per misurare resistenza e tempo."""
            self.reset()
            time.sleep(0.5)
            self.send_command(self.commands["zero_check_on"])
            self.send_command(self.commands["configure_resistance"])
            self.send_command(self.commands["format_elements"])
            
            # Opzionale: azzera il timestamp all'inizio dell'esperimento
            self.send_command(self.commands["reset_time"])
            
    """    
    def read_resistance_and_time(self):
        """Legge una misura di resistenza e il relativo timestamp."""
        # Se devi fare letture continue in un ciclo `while`, 
        # ti consiglio di spostare zero_check_off fuori da questa funzione.
        self.send_command(self.commands["zero_check_off"])
        time.sleep(0.1) 
        
        raw_value = self.query("READ?")
        
        self.send_command(self.commands["zero_check_on"])
        
        try:
            # Ora il Keithley restituirà qualcosa tipo: "+1.00000E+06OHM, +012.345secs"
            parts = raw_value.split(',')
            
            # Puliamo le stringhe
            res_str = parts[0].replace('OHM', '').replace('A', '').replace('V', '')
            time_str = parts[1].replace('secs', '')
            
            return float(res_str), float(time_str)
            
        except (ValueError, IndexError):
            return raw_value  # Ritorna la stringa grezza in caso di errore
    """


In [ ]:
class ProtocolloMaxtek:
    """Base class to handle the Maxtek binary packet protocol."""
    
    # Fixed header for all packets as per the manual
    SYNC = b'\xff\xfe' 
    
    def __init__(self, port, baudrate=9600, timeout=2):
        self.port = port
        self.baudrate = baudrate
        self.timeout = timeout
        self.serial = serial.Serial(self.port, self.baudrate, timeout=self.timeout)
        self.reset_buffers()

    def reset_buffers(self):
        self.serial.reset_input_buffer()
        self.serial.reset_output_buffer()

    def _compute_checksum(self, length, msg_type, data_bytes):
        """
        Compute the checksum as required by the manual:
        The remainder (modulo 256) of the 2's complement of the sum of Length, Type, and Data.
        """
        somma = length + msg_type + sum(data_bytes)
        checksum = (256 - (somma % 256)) % 256
        return checksum
    
    def _compute_checksum_1(self, length, code_number, data_bytes):
          
        somma = code_number + length + sum(data_bytes)
        
        # Prendi solo l'ultimo byte della somma (MOD 256)
        somma_byte = somma % 256
        
        # Calcola il complemento (NOT bit a bit)
        # In Python, questo si può fare con 255 - valore
        checksum = (255 - somma_byte) & 0xFF
        
        return checksum
        

    def send_packet(self, length, code_number, data_bytes, address):
        
        
        """
        Constructs and sends a packet following the specifications in the manual.
        """
        # "As shipped the Monitor is set up for an interface address of 1" (Chr$(1))
        # Secondo il manuale, LENGTH = byte da Message Type a Checksum inclusi.
        # Non credo, length è il numero di byte di istruzione che mando al monitor;
        # ad esempio, se scrivo 255+254+1+5(code number)+2(length)+x+x(questi 2 x sono i bytes a cui si riferisce lenght)+checksum
        
        length = len(data_bytes) #num_data_bytes
        
        checksum = self._compute_checksum_1(length, code_number, data_bytes) #questa usa il complemento a 2
        
        packet = self.SYNC + bytes([address & 0xFF]) + bytes([code_number & 0xFF]) + bytes([length & 0xFF]) + data_bytes + bytes([checksum])
        
        self.serial.write(packet) 

    def send_raw(self, raw_bytes):
        """
        Sends raw bytes directly. Useful for hardcoded commands
        that we know already work.
        """
        self.serial.write(raw_bytes)

    def close(self):
        if self.serial.is_open:
            self.serial.close()

    def __delete__(self, instance):
        self.close()

class BilanciaMaxtek(ProtocolloMaxtek):
    """Specific implementation for the Maxtek TM-350/400 scale."""
    
    # Manteniamo i comandi originali testati. Nota: sono già in formato bytes, quindi non serve convertirli ulteriormente.
    comandi_testati = {
        "identificazione": b'\xff\xfe\x01\x01\x00\xfe',
        "iniziazione_misure": b'\xff\xfe\x01\x05\x02\xf8\x01\xff',
        "stop": b'\xff\xfe\x01\x05\x02\x00\x00\xf8' 
    }

    def __init__(self, port, baudrate=9600, timeout=2):
        super().__init__(port, baudrate, timeout)

    def identify(self):
        self.send_raw(self.comandi_testati["identificazione"])
        response = self.serial.read(60)
        return response.decode('ascii', errors='ignore').strip()
    
    def start_measurements(self):
        self.send_raw(self.comandi_testati["iniziazione_misure"])
        self.serial.read(8)
    
    def read_measurement(self):
        """
        Read a single measurement packet and parse out the rate and thickness.
        """
        header = self.serial.read(5)
        if not header: # Se si pianta
            return None, None
            
        rate_string = self.serial.read(5) 
        thickness_string = self.serial.read(6)    
        tail = self.serial.read(32)

        try:
            rate = float(rate_string.decode('ascii').replace(",", "."))
            thickness = float(thickness_string.decode('ascii').replace(",", "."))
            #return rate, thickness
        except ValueError:
            return rate_string, thickness_string  # Return raw strings for debugging if parsing fails
    
    def stop_measurements(self):
        self.reset_buffers()
        self.send_raw(self.comandi_testati["stop"])
        self.reset_buffers()
        self.close()


In [ ]:
if False:
    #rm = pyvisa.ResourceManager()
    #inst = rm.list_resources()  #chiediamo quali porte sono aperte

    coms = list_ports.comports()
    #print("\nPorte seriali disponibili:")
    #[print(i, end='; ') for i in coms]

    bilancia = Bilancia(porta='/dev/ttyUSB0')
    print("Identificazione bilancia:", bilancia.identify())
    bilancia.reset_buffers()
    """
    bilancia.start_measurements()
    
    

    num_measurements = 10
    for i in range(num_measurements):
        print(f"Misura {i+1}/{num_measurements}:", bilancia.read_measurement())

    #print("Misura di prova:", bilancia.read_measurement())
    bilancia.stop_measurements()

"""

''

In [39]:
if False:
    camera = Camera(camera_index=2)
    camera.set_camera_params()
    im0 = camera.acquire_reference_image(avgs=16)

    plt.figure()
    plt.imshow(im0, cmap='gray')
    plt.colorbar()
    plt.show()

    camera.release()

""

''

In [57]:
coms = list_ports.comports()
print(*coms)

porta = "/dev/ttyUSB0"
baudrate = 9600
libra = serial.Serial(porta, baudrate, timeout=2)

#libra.write((r"*IDN?" + chr(10)).encode('latin-1'))
#libra.read(60)

libra.write((r":READ?" + chr(10)).encode('latin-1'))
print(libra.read(60))


/dev/ttyS4 - n/a /dev/ttyS31 - n/a /dev/ttyS30 - n/a /dev/ttyS29 - n/a /dev/ttyS28 - n/a /dev/ttyS27 - n/a /dev/ttyS26 - n/a /dev/ttyS25 - n/a /dev/ttyS24 - n/a /dev/ttyS23 - n/a /dev/ttyS22 - n/a /dev/ttyS21 - n/a /dev/ttyS20 - n/a /dev/ttyS19 - n/a /dev/ttyS18 - n/a /dev/ttyS17 - n/a /dev/ttyS16 - n/a /dev/ttyS15 - n/a /dev/ttyS14 - n/a /dev/ttyS13 - n/a /dev/ttyS12 - n/a /dev/ttyS11 - n/a /dev/ttyS10 - n/a /dev/ttyS9 - n/a /dev/ttyS8 - n/a /dev/ttyS7 - n/a /dev/ttyS6 - n/a /dev/ttyS5 - n/a /dev/ttyS3 - n/a /dev/ttyS2 - n/a /dev/ttyS1 - n/a /dev/ttyS0 - n/a /dev/ttyUSB0 - UC232R - UC232R
b'+9.910000E+37ZADC,+0001115.313078secs,+07269rdng#\r\n'


In [41]:
if True:
    rm = pyvisa.ResourceManager()
    risorse_collegate = rm.list_resources()

    print("Strumenti rilevati dal PC:")
    for risorsa in risorse_collegate:
        if "usb" in risorsa.lower():
            print(f"- {risorsa}")

Strumenti rilevati dal PC:


In [42]:
if True:
    multimetro = Multimetro(resource_manager=pyvisa.ResourceManager(), resource_name='ASRL/dev/ttyUSB1::INSTR')
    
    multimetro.multimeter.read()

    print("Versione firmware:", multimetro.query_version())
    
    #multimetro.close()

SerialException: [Errno 2] could not open port /dev/ttyUSB1: [Errno 2] No such file or directory: '/dev/ttyUSB1'

In [ ]:
# --- 1. CONNESSIONE STRUMENTI ---
print("Connessione agli strumenti...")
# Ricordati di mettere le porte giuste per Ubuntu (/dev/ttyUSB...)
libra = BilanciaMaxtek(port='/dev/ttyUSB0')
multimeter = ElettrometroKeithley(port='/dev/ttyUSB1')
camera = Camera(camera_index=0)

# --- 2. IDENTIFICAZIONE E SETUP ---
print("Identificazione Bilancia:", libra.identify())
multimeter.init_current_reading()

# Setup della fotocamera e presa dell'immagine di riferimento
print("Calibrazione camera in corso...")
camera.set_camera_params()
im0 = camera.acquire_reference_image(avgs=16)
h, w = im0.shape
masks = camera.build_roi_masks(im0, center_x=w//2, center_y=h//2, radius=50)

# --- 3. PREPARAZIONE PRESA DATI ---
libra.start_measurements()
time.sleep(1) # Diamo 1 secondo alla bilancia per stabilizzarsi

libra_data = []
multi_data = []
camera_data = []

startime = time.time()
print("\n--- MISURE IN CORSO ---")
print("Clicca sul terminale (o usa il tasto Stop di Jupyter) e premi CTRL+C per terminare.")

try:
    while True:
        # Calcolo il tempo esatto di questo ciclo
        t_attuale = round(time.time() - startime, 5)
        
        # LEGGO BILANCIA
        rate, spessore = libra.read_measurement()
        if rate is not None:
            libra_data.append((t_attuale, rate, spessore))
        
        # LEGGO MULTIMETRO
        misura = multimeter.read_current()
        multi_data.append((t_attuale, misura))
        
        # LEGGO CAMERA (prendiamo il parametro 'ii' come intensità totale)
        frame, heatmap, ii, ii_in, ii_mid, ii1, ii2, ii3, ii4 = camera.process_frame(im0, masks)
        camera_data.append((t_attuale, ii))
        
        # STAMPO A SCHERMO PER CONTROLLO
        print(f"T: {t_attuale}s | Spess: {spessore} | Corr/Res: {misura} | Luce: {round(ii, 2)}")
        
        # Una micro-pausa per non sovraccaricare le porte seriali
        time.sleep(0.1)

# Questa è l'eccezione che scatta se premi CTRL+C o "Stop"
except KeyboardInterrupt:
    print("\nFine delle misure richiesta dall'utente.")
except Exception as e:
    print(f"\nErrore imprevisto durante la misura: {e}")

# --- 4. SALVATAGGIO E CHIUSURA (Garantito) ---
finally:
    print("Salvataggio dei dati e chiusura porte...")
    
    # Salvo i dati della bilancia
    if len(libra_data) > 0:
        np.savetxt(fname="libra_data_sync.txt", X=libra_data, fmt='%s', delimiter='\t', header="Tempo\tRate\tSpessore")
    
    # Salvo i dati del multimetro
    if len(multi_data) > 0:
        np.savetxt(fname="multimeter_data_sync.txt", X=multi_data, fmt='%s', delimiter='\t', header="Tempo\tMisura")
        
    # Salvo i dati della camera
    if len(camera_data) > 0:
        np.savetxt(fname="camera_data_sync.txt", X=camera_data, fmt='%s', delimiter='\t', header="Tempo\tIntensita_Luce")
    
    # Spengo gli strumenti (usando i comandi delle classi)
    libra.stop_measurements()
    multimeter.close()
    camera.release()
    
    print("Tutto chiuso e salvato correttamente!")

In [ ]:
import matplotlib.pyplot as plt

# 1. CARICAMENTO DEI DATI
# Assicurati che i nomi dei file siano quelli corretti salvati in precedenza
# skiprows=1 serve per saltare la riga di testo dell'intestazione (Header)
try:
    dati_bil = np.loadtxt("libra_data_sync.txt", skiprows=1)
    dati_mult = np.loadtxt("multimeter_data_sync.txt", skiprows=1)
    
    # 2. ESTRAZIONE DELLE COLONNE
    # Bilancia: colonna 0 (Tempo), colonna 2 (Spessore)
    t_bil = dati_bil[:, 0]
    spessore = dati_bil[:, 2]
    
    # Multimetro: colonna 0 (Tempo), colonna 1 (Resistenza/Corrente)
    t_mult = dati_mult[:, 0]
    resistenza = dati_mult[:, 1]
    
    # 3. ALLINEAMENTO TEMPORALE (INTERPOLAZIONE)
    # Calcoliamo qual era il valore della resistenza nell'esatto istante in cui 
    # la bilancia ha misurato lo spessore.
    resistenza_sincronizzata = np.interp(t_bil, t_mult, resistenza)
    
    # 4. CREAZIONE DEL GRAFICO
    plt.figure(figsize=(10, 6))
    
    # Plottiamo Spessore (X) vs Resistenza (Y)
    plt.plot(spessore, resistenza_sincronizzata, '-', color='b', linewidth=2, label='Dati misurati')
    plt.plot(spessore, resistenza_sincronizzata, 'o', color='red', markersize=4) # Aggiunge i puntini
    
    # Personalizzazione del grafico
    plt.title("Resistenza vs Spessore", fontsize=14)
    plt.xlabel("Spessore", fontsize=12)
    plt.ylabel("Resistenza", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.yscale('log')
    
    plt.legend()
    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print("Errore: I file .txt non sono stati trovati. Assicurati di aver fatto prima un'acquisizione!")